# Studi Kasus Integrasi: Automated Book Intelligence System

In [41]:
import requests
import json
import csv
from bs4 import BeautifulSoup
from datetime import datetime

url = "http://books.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
}

rating_sample = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}
daftar_buku = []

# scraping data
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Terjadi kesalahan pada jaringan: {e}")
    response = None
    
if response:
    soup = BeautifulSoup(response.text, 'html.parser')
    semua_buku = soup.find_all('article', class_='product_pod')
    
    for buku in semua_buku:
        judul = buku.find('h3').find('a')['title']
        harga = float(buku.find('p', class_='price_color').text.replace("Â£",""))
        rating = buku.find('p', class_='star-rating')['class'][1]
        waktu_scraped = datetime.now()
    
        baris_buku = {
            'judul': judul,
            'harga': harga,
            'rating': rating_sample[rating],
            'waktu_scraped': waktu_scraped.strftime("%Y-%m-%d %H:%M:%S")
        }
        daftar_buku.append(baris_buku)
    print("Data berhasil di ambil dan tersimpan di daftar_buku")
    
    # Hitung statistik
    daftar_semua_harga = [buku['harga'] for buku in daftar_buku]
    tertinggi = max(daftar_buku, key= lambda x: x['harga'])
    terendah = min(daftar_buku, key= lambda x: x['harga']) 
    jumlah_buku = len(daftar_buku)
    rata_rata = sum(daftar_semua_harga) / jumlah_buku
    
    print(f"\n-- STATISTIK DAFTAR BUKU --")
    print(f"Jumlah buku       : {jumlah_buku}")
    print(f"Harga rata-rata   : {rata_rata:.2f}")
    print(f"Harga tertinggi   : {tertinggi['judul']} | {tertinggi['harga']}")
    print(f"Harga terendah    : {terendah['judul']} | {terendah['harga']}")

    # Laporan untuk JSON file
    laporan = {
        'waktu_laporan': datetime.now().strftime("%d %B %Y, Jam %H:%M WIB"),
        'total_buku': jumlah_buku,
        'rata_rata_harga': rata_rata,
        'buku_termahal': {
            'Judul': tertinggi['judul'],
            'Harga': tertinggi['harga']
        },
        'buku_termurah': {
            'Judul': terendah['judul'],
            'Harga': terendah['harga']
        }
    }

    # Menulis atau menyimpan data ke file csv dan laporan ke file json
    try:
        with open('katalog_buku.csv', 'w', newline='', encoding='utf-8') as file_csv, \
            open('laporan_eksekutif.json', 'w', encoding='utf-8') as file_json:
            header = ['judul', 'harga', 'rating', 'waktu_scraped']
            writer = csv.DictWriter(file_csv, fieldnames=header)
            writer.writeheader()
            writer.writerows(daftar_buku)
            json.dump(laporan, file_json, indent=4)
        print("Data berhasil disimpan ke file")
    except IOError as e:
        print(f"Gagal menyimpan file: {e}")

Data berhasil di ambil dan tersimpan di daftar_buku

-- STATISTIK DAFTAR BUKU --
Jumlah buku       : 20
Harga rata-rata   : 38.05
Harga tertinggi   : Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 | 57.25
Harga terendah    : Starving Hearts (Triangular Trade Trilogy, #1) | 13.99
Data berhasil disimpan ke file
